In [90]:
%load_ext autoreload
%autoreload 2
import os
import torch
import numpy as np
import json



from compactreasoningmodels.datasets import NonogramDataset
from compactreasoningmodels.datasets.collate import collate_raw

if 'original_dir' not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
os.environ["DATA_DIR"] = os.path.join(os.getcwd(), "data")
os.environ["MODEL_DIR"] = os.path.join(os.getcwd(), "models")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [91]:
dataset = NonogramDataset("traces/nonogram_5x5.jsonl")
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_raw)

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd

results = defaultdict(lambda: {"accuracy": [], "mse_loss": [], "cr_loss": []})

for clues, grid, meta in dataloader:
    true_grid = grid[0]
    traces = meta[0].get("traces", None)
    if not traces:
        continue

    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            key = (trace_name, sampling_ratio)
            results[key]["accuracy"].append(np.mean([run["solved"] for run in runs]))
            results[key]["mse_loss"].append(np.mean([run["mse_losses"][-1] for run in runs]))
            results[key]["cr_loss"].append(np.mean([run["cr_losses"][-1] for run in runs]))

# Build a tidy summary table: mean ± std for each metric
rows = []
for (trace_name, sampling_ratio), metrics in results.items():
    row = {"trace": trace_name, "sampling_ratio": sampling_ratio}
    for metric_name, values in metrics.items():
        row[f"{metric_name}_mean"] = np.mean(values)
        row[f"{metric_name}_std"] = np.std(values)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(["trace", "sampling_ratio"]).reset_index(drop=True)
df

In [117]:
from collections import defaultdict
from itertools import product
import numpy as np
import pandas as pd

# metric_name -> (run_key, how to extract the value from run[run_key])
metric_extractors = {
    "accuracy": ("solved", lambda v: v),              # scalar per run
    "mse_loss": ("mse_losses", lambda v: v[-1]),       # last-step loss
    "cr_loss": ("cr_losses", lambda v: v[-1]),         # last-step loss
}
metrics = list(metric_extractors.keys())

data = defaultdict(lambda: defaultdict(list))

for clues, grid, meta in dataloader:
    traces = meta[0].get("traces", None)
    if not traces:
        continue
    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            if not runs:
                continue
            key = (trace_name, sampling_ratio)
            for metric_name, (run_key, extract) in metric_extractors.items():
                run_values = [extract(run[run_key]) for run in runs]
                data[key][metric_name].append(run_values)

keys = sorted(data.keys())
key_labels = {k: f"{k[0]}|{k[1]}" for k in keys}

def paired_values(key_a, key_b, metric):
    a_vals, b_vals = [], []
    for runs_a, runs_b in zip(data[key_a][metric], data[key_b][metric]):
        for i, va in enumerate(runs_a):
            for j, vb in enumerate(runs_b):
                if key_a == key_b and i == j:
                    continue
                a_vals.append(va)
                b_vals.append(vb)
    return np.array(a_vals), np.array(b_vals)

labels = [key_labels[k] for k in keys]
corr_matrices = {
    metric: pd.DataFrame(index=labels, columns=labels, dtype=float)
    for metric in metrics
}

for metric in metrics:
    for k1, k2 in product(keys, keys):
        a, b = paired_values(k1, k2, metric)
        if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
            val = np.nan
        else:
            val = np.corrcoef(a, b)[0, 1]
        corr_matrices[metric].loc[key_labels[k1], key_labels[k2]] = val

for metric in metrics:
    print(f"Correlation matrix ({metric}):")
    display(corr_matrices[metric])

Correlation matrix (accuracy):


,genetic_algorithm_det|0.5,genetic_algorithm_det|0.75,genetic_algorithm_det|1.0,gradient_descent_global_adam|0.5,gradient_descent_global_adam|0.75,gradient_descent_global_adam|1.0,mac|0.5,mac|0.75,mac|1.0,model_solver|0.5,model_solver|0.75,model_solver|1.0
genetic_algorithm_det|0.5,0.366682,0.323623,0.421271,-0.024917,-0.032299,-0.032299,0.179947,0.103637,0.094414,0.152842,0.238660,0.242895
genetic_algorithm_det|0.75,0.323623,0.337616,0.435055,0.016901,0.021909,0.021909,0.198748,0.109555,0.099805,0.155665,0.228484,0.252853
genetic_algorithm_det|1.0,0.421271,0.435055,NaN,0.168345,0.218218,0.218218,0.183706,0.102658,0.093522,0.245592,0.326479,0.238095
gradient_descent_global_adam|0.5,-0.024917,0.016901,0.168345,0.493927,0.771454,0.771454,0.035914,0.017282,0.015744,0.023751,0.099285,0.072148
gradient_descent_global_adam|0.75,-0.032299,0.021909,0.218218,0.771454,1.000000,1.000000,0.046554,0.022402,0.020408,0.030787,0.128698,0.093522
gradient_descent_global_adam|1.0,-0.032299,0.021909,0.218218,0.771454,1.000000,NaN,0.046554,0.022402,0.020408,0.030787,0.128698,0.093522
mac|0.5,0.179947,0.198748,0.183706,0.035914,0.046554,0.046554,0.631268,0.445717,0.438380,0.100467,0.257971,0.468153
mac|0.75,0.103637,0.109555,0.102658,0.017282,0.022402,0.022402,0.445717,0.829235,0.911006,0.060080,0.174065,0.239535
mac|1.0,0.094414,0.099805,0.093522,0.015744,0.020408,0.020408,0.438380,0.911006,NaN,0.054733,0.158574,0.218218
model_solver|0.5,0.152842,0.155665,0.245592,0.023751,0.030787,0.030787,0.100467,0.060080,0.054733,0.157827,0.171808,0.172437


Correlation matrix (mse_loss):


,genetic_algorithm_det|0.5,genetic_algorithm_det|0.75,genetic_algorithm_det|1.0,gradient_descent_global_adam|0.5,gradient_descent_global_adam|0.75,gradient_descent_global_adam|1.0,mac|0.5,mac|0.75,mac|1.0,model_solver|0.5,model_solver|0.75,model_solver|1.0
genetic_algorithm_det|0.5,0.651824,0.699062,0.731444,0.383754,0.404362,0.413944,0.279613,0.135150,0.116846,0.360012,0.397729,0.377060
genetic_algorithm_det|0.75,0.699062,0.769580,0.797064,0.394331,0.386406,0.402548,0.316896,0.170249,0.150092,0.337835,0.387304,0.372362
genetic_algorithm_det|1.0,0.731444,0.797064,NaN,0.436966,0.455317,0.464421,0.197261,-0.010804,-0.025342,0.336729,0.344161,0.334137
gradient_descent_global_adam|0.5,0.383754,0.394331,0.436966,0.850383,0.891192,0.907683,0.264478,0.263230,0.260604,0.268649,0.235595,0.173882
gradient_descent_global_adam|0.75,0.404362,0.386406,0.455317,0.891192,0.937179,0.961987,0.242618,0.208446,0.206295,0.305273,0.256895,0.196198
gradient_descent_global_adam|1.0,0.413944,0.402548,0.464421,0.907683,0.961987,NaN,0.284338,0.262402,0.260797,0.313663,0.289692,0.237721
mac|0.5,0.279613,0.316896,0.197261,0.264478,0.242618,0.284338,0.810389,0.728758,0.719369,0.290560,0.451063,0.515805
mac|0.75,0.135150,0.170249,-0.010804,0.263230,0.208446,0.262402,0.728758,0.981458,0.990724,0.151330,0.378833,0.448764
mac|1.0,0.116846,0.150092,-0.025342,0.260604,0.206295,0.260797,0.719369,0.990724,NaN,0.135477,0.359437,0.422671
model_solver|0.5,0.360012,0.337835,0.336729,0.268649,0.305273,0.313663,0.290560,0.151330,0.135477,0.327194,0.419996,0.432079


Correlation matrix (cr_loss):


,genetic_algorithm_det|0.5,genetic_algorithm_det|0.75,genetic_algorithm_det|1.0,gradient_descent_global_adam|0.5,gradient_descent_global_adam|0.75,gradient_descent_global_adam|1.0,mac|0.5,mac|0.75,mac|1.0,model_solver|0.5,model_solver|0.75,model_solver|1.0
genetic_algorithm_det|0.5,1.0,1.0,0.999660,1.0,1.0,0.999660,1.0,1.0,0.999660,1.0,1.0,0.999660
genetic_algorithm_det|0.75,1.0,1.0,0.999988,1.0,1.0,0.999988,1.0,1.0,0.999988,1.0,1.0,0.999988
genetic_algorithm_det|1.0,1.0,1.0,NaN,1.0,1.0,0.993697,1.0,1.0,0.993697,1.0,1.0,0.993697
gradient_descent_global_adam|0.5,1.0,1.0,0.999995,1.0,1.0,0.999995,1.0,1.0,0.999995,1.0,1.0,0.999995
gradient_descent_global_adam|0.75,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000
gradient_descent_global_adam|1.0,1.0,1.0,0.997572,1.0,1.0,NaN,1.0,1.0,0.997572,1.0,1.0,0.997572
mac|0.5,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000
mac|0.75,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000,1.0,1.0,1.000000
mac|1.0,1.0,1.0,0.990893,1.0,1.0,0.990893,1.0,1.0,NaN,1.0,1.0,0.990893
model_solver|0.5,1.0,1.0,0.996839,1.0,1.0,0.996839,1.0,1.0,0.996839,1.0,1.0,0.996839


In [ ]:
from collections import defaultdict
from itertools import product
import numpy as np
import pandas as pd

metric_key_map = {"accuracy": "solved", "mse_loss": "mse_losses", "cr_loss": "cr_losses"}
metrics = list(metric_key_map.keys())

data = defaultdict(lambda: defaultdict(list))

for clues, grid, meta in dataloader:
    traces = meta[0].get("traces", None)
    if not traces:
        continue
    for trace_name, sampling_ratios in traces.items():
        for sampling_ratio, runs in sampling_ratios.items():
            if not runs:
                continue
            key = (trace_name, sampling_ratio)
            for metric_name, run_key in metric_key_map.items():
                run_values = [np.mean(run[run_key]) for run in runs]
                data[key][metric_name].append(run_values)

keys = sorted(data.keys())  # tuples (trace_name, sampling_ratio)
key_labels = {k: f"{k[0]}|{k[1]}" for k in keys}  # string label per key

def paired_values(key_a, key_b, metric):
    a_vals, b_vals = [], []
    for runs_a, runs_b in zip(data[key_a][metric], data[key_b][metric]):
        for i, va in enumerate(runs_a):
            for j, vb in enumerate(runs_b):
                if key_a == key_b and i == j:
                    continue
                a_vals.append(va)
                b_vals.append(vb)
    return np.array(a_vals), np.array(b_vals)

labels = [key_labels[k] for k in keys]
corr_matrices = {
    metric: pd.DataFrame(index=labels, columns=labels, dtype=float)
    for metric in metrics
}

for metric in metrics:
    for k1, k2 in product(keys, keys):
        a, b = paired_values(k1, k2, metric)
        if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
            val = np.nan
        else:
            val = np.corrcoef(a, b)[0, 1]
        corr_matrices[metric].loc[key_labels[k1], key_labels[k2]] = val

for metric in metrics:
    print(f"Correlation matrix ({metric}):")
    display(corr_matrices[metric])